# 🐼 pandas Mastery — One Notebook to Rule Them All

A single, hands-on notebook from fundamentals to advanced pandas. Every section has a
**Concept**, a runnable **Demo**, and an **Exercise** with the solution right below.

**How to use:** run top to bottom. At each *Exercise*, try the empty cell before peeking.

## Table of Contents
1. Core objects: Series & DataFrame
2. Creating DataFrames
3. Inspecting data
4. Selecting & indexing (`loc`, `iloc`, `[]`)
5. Boolean filtering & `query`
6. Adding / modifying / dropping columns
7. Missing data
8. Dtypes & conversions
9. String operations (`.str`)
10. Sorting & ranking
11. GroupBy: split–apply–combine
12. Pivot tables & crosstab
13. Merge, join, concat
14. Reshaping: melt / pivot / stack / unstack
15. Time series
16. apply / map / transform
17. Window functions (rolling / expanding)
18. MultiIndex
19. Categorical data
20. File I/O
21. Performance
22. 🏆 Capstone

## 1. Core Objects: Series & DataFrame

- **Series** — a 1-D labeled array (a column).
- **DataFrame** — a 2-D labeled table (a dict of Series sharing an index).

The **index** is the row-label structure that makes alignment automatic.

In [1]:
import pandas as pd
import numpy as np
print("pandas", pd.__version__)

s = pd.Series([10, 20, 30], index=['a', 'b', 'c'], name='scores')
print(s)
print("value at 'b':", s['b'])

df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Cara'],
    'age': [30, 25, 35],
    'city': ['NYC', 'LA', 'SF'],
})
print(df)
print("index:", list(df.index), "| columns:", list(df.columns))

pandas 3.1.0
a    10
b    20
c    30
Name: scores, dtype: int64
value at 'b': 20
    name  age city
0  Alice   30  NYC
1    Bob   25   LA
2   Cara   35   SF
index: [0, 1, 2] | columns: ['name', 'age', 'city']


**Exercise 1** — Build a Series of the squares of 1..5, indexed by the numbers themselves. Print the square of 4 by label.

In [2]:
# Your turn:

In [3]:
# Solution
sq = pd.Series([n**2 for n in range(1, 6)], index=range(1, 6))
print(sq)
print("square of 4:", sq[4])

1     1
2     4
3     9
4    16
5    25
dtype: int64
square of 4: 16


## 2. Creating DataFrames

From dicts, lists of rows, NumPy arrays, or files. Most real work starts from a file (section 20).

In [4]:
# From a dict of columns
df1 = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

# From a list of row dicts
df2 = pd.DataFrame([{'x': 1, 'y': 4}, {'x': 2, 'y': 5}])

# From a NumPy array with explicit labels
df3 = pd.DataFrame(np.arange(6).reshape(2, 3),
                   columns=['a', 'b', 'c'], index=['r1', 'r2'])
print(df3)

# From records / parallel lists
names = ['a', 'b', 'c']; vals = [1, 2, 3]
print(pd.DataFrame({'name': names, 'val': vals}))

    a  b  c
r1  0  1  2
r2  3  4  5
  name  val
0    a    1
1    b    2
2    c    3


**Exercise 2** — Create a 3-row DataFrame of products with columns `sku`, `price`, `qty`. Add a computed column `revenue = price*qty`.

In [5]:
# Your turn:

In [6]:
# Solution
prods = pd.DataFrame({'sku': ['A1', 'B2', 'C3'],
                      'price': [9.99, 4.50, 19.0],
                      'qty': [10, 25, 4]})
prods['revenue'] = prods['price'] * prods['qty']
print(prods)

  sku  price  qty  revenue
0  A1   9.99   10     99.9
1  B2   4.50   25    112.5
2  C3  19.00    4     76.0


## 3. Inspecting Data

First thing you do with any new dataset.

In [7]:
rng = np.random.default_rng(0)
df = pd.DataFrame({
    'group': rng.choice(list('ABC'), 100),
    'value': rng.normal(50, 10, 100),
    'count': rng.integers(1, 20, 100),
})
print(df.head(3))          # first rows
print(df.tail(3))          # last rows
print("shape:", df.shape)
print(df.dtypes)
print(df.describe())       # numeric summary
df.info()                  # dtypes + non-null counts + memory

  group      value  count
0     C  53.573804      5
1     B  37.916814     13
2     B  49.955459     10
   group      value  count
97     A  48.910853      4
98     B  41.962682      7
99     C  60.801634     13
shape: (100, 3)
group        str
value    float64
count      int64
dtype: object
            value       count
count  100.000000  100.000000
mean    50.300326   10.880000
std      9.848058    5.379122
min     26.346961    1.000000
25%     43.254560    6.000000
50%     51.747644   11.000000
75%     57.035555   15.000000
max     70.023926   19.000000
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   group   100 non-null    str    
 1   value   100 non-null    float64
 2   count   100 non-null    int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 2.6 KB


**Exercise 3** — For the `df` above, print the number of unique groups and the mean `value` per the `.describe()` summary.

In [8]:
# Your turn:

In [9]:
# Solution
print("unique groups:", df['group'].nunique())
print("value mean:", df['value'].mean().round(3))

unique groups: 3
value mean: 50.3


## 4. Selecting & Indexing

- `df[col]` → a column (Series). `df[[c1, c2]]` → sub-DataFrame.
- `df.loc[rows, cols]` → **label**-based.
- `df.iloc[rows, cols]` → **integer position**-based.
- `.at` / `.iat` → fast scalar access.

In [10]:
df = pd.DataFrame({'a': [1, 2, 3, 4], 'b': [10, 20, 30, 40], 'c': [100, 200, 300, 400]},
                  index=['w', 'x', 'y', 'z'])
print(df['a'])              # column
print(df[['a', 'c']])       # two columns

print(df.loc['x'])          # row by label
print(df.loc['x', 'b'])     # scalar by label
print(df.loc['x':'y', ['a', 'b']])   # label slice is INCLUSIVE on the stop

print(df.iloc[1])           # row by position
print(df.iloc[0:2, 0:2])    # position slice is EXCLUSIVE on the stop
print(df.iat[0, 1])         # fast scalar

w    1
x    2
y    3
z    4
Name: a, dtype: int64
   a    c
w  1  100
x  2  200
y  3  300
z  4  400
a      2
b     20
c    200
Name: x, dtype: int64
20
   a   b
x  2  20
y  3  30
a      2
b     20
c    200
Name: x, dtype: int64
   a   b
w  1  10
x  2  20
10


**Exercise 4** — From the `df` above, select rows `x` and `z`, columns `a` and `c`, using `.loc`.

In [11]:
# Your turn:

In [12]:
# Solution
print(df.loc[['x', 'z'], ['a', 'c']])

   a    c
x  2  200
z  4  400


## 5. Boolean Filtering & `query`

Filter rows with a boolean mask, or the readable `.query()` string.

In [13]:
rng = np.random.default_rng(1)
df = pd.DataFrame({'name': list('abcdef'),
                   'age': rng.integers(18, 60, 6),
                   'salary': rng.integers(30, 120, 6) * 1000})
print(df)
print(df[df['age'] > 30])
print(df[(df['age'] > 30) & (df['salary'] < 100000)])   # &, |, ~ with parens
print(df.query('age > 30 and salary < 100000'))         # same, as a string
print(df[df['name'].isin(['a', 'c', 'e'])])

  name  age  salary
0    a   37  104000
1    b   39  115000
2    c   49   52000
3    d   57   58000
4    e   19  108000
5    f   24   68000
  name  age  salary
0    a   37  104000
1    b   39  115000
2    c   49   52000
3    d   57   58000
  name  age  salary
2    c   49   52000
3    d   57   58000
  name  age  salary
2    c   49   52000
3    d   57   58000
  name  age  salary
0    a   37  104000
2    c   49   52000
4    e   19  108000


**Exercise 5** — Select rows where `salary` is above its own mean.

In [14]:
# Your turn:

In [15]:
# Solution
print(df[df['salary'] > df['salary'].mean()])

  name  age  salary
0    a   37  104000
1    b   39  115000
4    e   19  108000


## 6. Adding / Modifying / Dropping Columns

`assign` returns a new frame; direct assignment mutates in place. `drop` removes rows or columns.

In [16]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
df['c'] = df['a'] + df['b']          # new column
df['a'] = df['a'] * 10               # modify in place
df2 = df.assign(d=lambda x: x['c'] ** 2)   # chainable, returns new frame
print(df2)

print(df.drop(columns=['b']))        # drop a column
print(df.drop(index=[0]))            # drop a row
df = df.rename(columns={'a': 'A'})   # rename
print(df.columns.tolist())

    a  b  c   d
0  10  4  5  25
1  20  5  7  49
2  30  6  9  81
    a  c
0  10  5
1  20  7
2  30  9
    a  b  c
1  20  5  7
2  30  6  9
['A', 'b', 'c']


**Exercise 6** — Given a frame with `celsius` temps, add a `fahrenheit` column (C*9/5+32), then drop `celsius`.

In [17]:
# Your turn:

In [18]:
# Solution
t = pd.DataFrame({'celsius': [0, 25, 37, 100]})
t['fahrenheit'] = t['celsius'] * 9/5 + 32
t = t.drop(columns='celsius')
print(t)

   fahrenheit
0        32.0
1        77.0
2        98.6
3       212.0


## 7. Missing Data

`NaN` marks missing. Detect with `isna`, remove with `dropna`, fill with `fillna`/`interpolate`.

In [19]:
df = pd.DataFrame({'a': [1, np.nan, 3, np.nan],
                   'b': [np.nan, 2, 3, 4],
                   'c': [1, 2, 3, 4]})
print(df.isna())
print("nulls per column:\n", df.isna().sum())

print("dropna rows:\n", df.dropna())            # drop any row with a NaN
print("fill 0:\n", df.fillna(0))
print("fill col mean:\n", df.fillna(df.mean()))
print("ffill:\n", df.ffill())                    # carry last value forward
print("interpolate:\n", df.interpolate())

       a      b      c
0  False   True  False
1   True  False  False
2  False  False  False
3   True  False  False
nulls per column:
 a    2
b    1
c    0
dtype: int64
dropna rows:
      a    b  c
2  3.0  3.0  3
fill 0:
      a    b  c
0  1.0  0.0  1
1  0.0  2.0  2
2  3.0  3.0  3
3  0.0  4.0  4
fill col mean:
      a    b  c
0  1.0  3.0  1
1  2.0  2.0  2
2  3.0  3.0  3
3  2.0  4.0  4
ffill:
      a    b  c
0  1.0  NaN  1
1  1.0  2.0  2
2  3.0  3.0  3
3  3.0  4.0  4
interpolate:
      a    b  c
0  1.0  NaN  1
1  2.0  2.0  2
2  3.0  3.0  3
3  3.0  4.0  4


**Exercise 7** — Fill missing values in each column with that column's median, and report how many were filled.

In [20]:
# Your turn:

In [21]:
# Solution
df = pd.DataFrame({'x': [1, np.nan, 3, 4, np.nan], 'y': [np.nan, 2, 2, 2, 10]})
n_missing = df.isna().sum().sum()
filled = df.fillna(df.median())
print(filled)
print("values filled:", int(n_missing))

     x     y
0  1.0   2.0
1  3.0   2.0
2  3.0   2.0
3  4.0   2.0
4  3.0  10.0
values filled: 3


## 8. Dtypes & Conversions

Right dtypes save memory and unlock the correct operations (dates, categories).

In [22]:
df = pd.DataFrame({'n': ['1', '2', '3'], 'd': ['2023-01-01', '2023-06-15', '2023-12-31']})
print(df.dtypes)
df['n'] = df['n'].astype(int)
df['d'] = pd.to_datetime(df['d'])
print(df.dtypes)
print(df['d'].dt.month.tolist())        # datetime accessor
# Downcast / category to shrink memory
big = pd.Series(['red', 'green', 'red', 'blue'] * 1000)
print("object bytes :", big.memory_usage(deep=True))
print("category bytes:", big.astype('category').memory_usage(deep=True))

n    str
d    str
dtype: object
n             int64
d    datetime64[us]
dtype: object
[1, 6, 12]
object bytes : 47132
category bytes: 4169


**Exercise 8** — Convert a column of `'$1,200'`-style strings into integers (strip `$` and `,`).

In [23]:
# Your turn:

In [24]:
# Solution
s = pd.Series(['$1,200', '$3,450', '$900'])
nums = s.str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(int)
print(nums)

0    1200
1    3450
2     900
dtype: int64


## 9. String Operations (`.str`)

Vectorized string methods on object/string columns.

In [25]:
s = pd.Series(['  Alice ', 'BOB', 'cara', 'Dan_Smith'])
print(s.str.strip().str.title())
print(s.str.lower())
print(s.str.len())
print(s.str.contains('a', case=False))
print(s.str.replace('_', ' ', regex=False))
print(pd.Series(['a-1', 'b-2']).str.split('-', expand=True))   # into columns
print(s.str.extract(r'([A-Za-z]+)'))                            # regex capture

0        Alice
1          Bob
2         Cara
3    Dan_Smith
dtype: str
0       alice 
1          bob
2         cara
3    dan_smith
dtype: str
0    8
1    3
2    4
3    9
dtype: int64
0     True
1    False
2     True
3     True
dtype: bool
0       Alice 
1          BOB
2         cara
3    Dan Smith
dtype: str
   0  1
0  a  1
1  b  2
       0
0  Alice
1    BOB
2   cara
3    Dan


**Exercise 9** — From emails like `'alice@acme.com'`, extract just the domain (`acme.com`).

In [26]:
# Your turn:

In [27]:
# Solution
emails = pd.Series(['alice@acme.com', 'bob@globex.io', 'cara@initech.net'])
print(emails.str.split('@').str[1])

0       acme.com
1      globex.io
2    initech.net
dtype: object


## 10. Sorting & Ranking

In [28]:
df = pd.DataFrame({'name': list('abcde'), 'score': [88, 72, 95, 72, 60]})
print(df.sort_values('score', ascending=False))
print(df.sort_values(['score', 'name']))        # multi-key
print(df.sort_index())
df['rank'] = df['score'].rank(ascending=False, method='min').astype(int)
print(df)
print("nlargest 2:\n", df.nlargest(2, 'score'))

  name  score
2    c     95
0    a     88
1    b     72
3    d     72
4    e     60
  name  score
4    e     60
1    b     72
3    d     72
0    a     88
2    c     95
  name  score
0    a     88
1    b     72
2    c     95
3    d     72
4    e     60
  name  score  rank
0    a     88     2
1    b     72     3
2    c     95     1
3    d     72     3
4    e     60     5
nlargest 2:
   name  score  rank
2    c     95     1
0    a     88     2


**Exercise 10** — Sort a sales frame by region (A→Z) then by sales (high→low) within each region.

In [29]:
# Your turn:

In [30]:
# Solution
sales = pd.DataFrame({'region': ['B', 'A', 'B', 'A'], 'sales': [10, 30, 25, 5]})
print(sales.sort_values(['region', 'sales'], ascending=[True, False]))

  region  sales
1      A     30
3      A      5
2      B     25
0      B     10


## 11. GroupBy: Split–Apply–Combine

The workhorse of analysis: split rows into groups, apply a function, combine results.

In [31]:
rng = np.random.default_rng(2)
df = pd.DataFrame({'dept': rng.choice(['eng', 'sales', 'hr'], 20),
                   'salary': rng.integers(50, 150, 20) * 1000,
                   'age': rng.integers(22, 60, 20)})
print(df.groupby('dept')['salary'].mean())
print(df.groupby('dept').agg(
    avg_salary=('salary', 'mean'),
    max_age=('age', 'max'),
    headcount=('salary', 'size'),
))
# transform broadcasts the group result back to original rows
df['salary_vs_dept_mean'] = df['salary'] - df.groupby('dept')['salary'].transform('mean')
print(df.head())

dept
eng       96125.0
hr       106000.0
sales    105500.0
Name: salary, dtype: float64
       avg_salary  max_age  headcount
dept                                 
eng       96125.0       51          8
hr       106000.0       57          6
sales    105500.0       59          6
    dept  salary  age  salary_vs_dept_mean
0     hr   80000   34             -26000.0
1    eng  106000   35               9875.0
2    eng   76000   44             -20125.0
3    eng   65000   41             -31125.0
4  sales  124000   48              18500.0


**Exercise 11** — For each department, compute the count and the salary range (max − min).

In [32]:
# Your turn:

In [33]:
# Solution
out = df.groupby('dept')['salary'].agg(['count', lambda x: x.max() - x.min()])
out.columns = ['count', 'salary_range']
print(out)

       count  salary_range
dept                      
eng        8         93000
hr         6         72000
sales      6         76000


## 12. Pivot Tables & Crosstab

Spreadsheet-style summaries: aggregate a value across two categorical axes.

In [34]:
rng = np.random.default_rng(3)
df = pd.DataFrame({'region': rng.choice(['N', 'S'], 40),
                   'product': rng.choice(['x', 'y', 'z'], 40),
                   'sales': rng.integers(1, 100, 40)})
print(pd.pivot_table(df, values='sales', index='region', columns='product',
                     aggfunc='sum', fill_value=0, margins=True))
print(pd.crosstab(df['region'], df['product']))   # frequency counts

product    x    y    z   All
region                      
N        438  528  223  1189
S        497  275  169   941
All      935  803  392  2130
product   x  y  z
region           
N         7  8  6
S        10  5  4


**Exercise 12** — Build a pivot of mean `sales` by `product` (rows) and `region` (columns).

In [35]:
# Your turn:

In [36]:
# Solution
print(pd.pivot_table(df, values='sales', index='product', columns='region', aggfunc='mean').round(1))

region      N     S
product            
x        62.6  49.7
y        66.0  55.0
z        37.2  42.2


## 13. Merge, Join, Concat

Combine tables. `merge` = SQL-style joins on keys; `concat` = stack frames.

In [37]:
customers = pd.DataFrame({'cid': [1, 2, 3], 'name': ['Alice', 'Bob', 'Cara']})
orders = pd.DataFrame({'cid': [1, 1, 2, 4], 'amount': [100, 50, 70, 30]})

print(pd.merge(customers, orders, on='cid', how='inner'))
print(pd.merge(customers, orders, on='cid', how='left'))
print(pd.merge(customers, orders, on='cid', how='outer', indicator=True))

a = pd.DataFrame({'x': [1, 2]}); b = pd.DataFrame({'x': [3, 4]})
print(pd.concat([a, b], ignore_index=True))       # stack rows
print(pd.concat([a, b], axis=1))                    # side by side

   cid   name  amount
0    1  Alice     100
1    1  Alice      50
2    2    Bob      70
   cid   name  amount
0    1  Alice   100.0
1    1  Alice    50.0
2    2    Bob    70.0
3    3   Cara     NaN
   cid   name  amount      _merge
0    1  Alice   100.0        both
1    1  Alice    50.0        both
2    2    Bob    70.0        both
3    3   Cara     NaN   left_only
4    4    NaN    30.0  right_only
   x
0  1
1  2
2  3
3  4
   x  x
0  1  3
1  2  4


**Exercise 13** — A left-join of customers to orders, then total amount per customer (0 for those with no orders).

In [38]:
# Your turn:

In [39]:
# Solution
merged = pd.merge(customers, orders, on='cid', how='left')
totals = merged.groupby('name')['amount'].sum().fillna(0)
print(totals)

name
Alice    150.0
Bob       70.0
Cara       0.0
Name: amount, dtype: float64


## 14. Reshaping: melt / pivot / stack / unstack

- **melt**: wide → long (columns become rows).
- **pivot**: long → wide (inverse of melt).
- **stack/unstack**: move the innermost column level to/from the index.

In [40]:
wide = pd.DataFrame({'id': [1, 2],
                     'math': [90, 80],
                     'science': [85, 95]})
long = wide.melt(id_vars='id', var_name='subject', value_name='score')
print(long)
back = long.pivot(index='id', columns='subject', values='score')
print(back)
print(back.stack())      # columns -> index level (Series)

   id  subject  score
0   1     math     90
1   2     math     80
2   1  science     85
3   2  science     95
subject  math  science
id                    
1          90       85
2          80       95
id  subject
1   math       90
    science    85
2   math       80
    science    95
dtype: int64


**Exercise 14** — Take a wide quarterly-sales frame (`q1..q4` columns per store) and melt it to long form with `quarter` and `sales` columns.

In [41]:
# Your turn:

In [42]:
# Solution
w = pd.DataFrame({'store': ['A', 'B'], 'q1': [10, 20], 'q2': [15, 25],
                  'q3': [12, 22], 'q4': [18, 28]})
print(w.melt(id_vars='store', var_name='quarter', value_name='sales'))

  store quarter  sales
0     A      q1     10
1     B      q1     20
2     A      q2     15
3     B      q2     25
4     A      q3     12
5     B      q3     22
6     A      q4     18
7     B      q4     28


## 15. Time Series

A datetime index unlocks resampling, rolling windows, and date-aware slicing.

In [43]:
idx = pd.date_range('2023-01-01', periods=90, freq='D')
rng = np.random.default_rng(4)
ts = pd.Series(rng.normal(0, 1, 90).cumsum() + 100, index=idx)
print(ts.head(3))
print("Jan mean:", ts['2023-01'].mean().round(2))          # partial-string slice
print("weekly mean:\n", ts.resample('W').mean().head(3))   # downsample
print("7-day rolling mean:\n", ts.rolling(7).mean().head(8))
print(ts.index.day_name()[:3].tolist())                     # dt components on index

2023-01-01     99.348209
2023-01-02     99.173492
2023-01-03    100.837216
Freq: D, dtype: float64
Jan mean: 99.01
weekly mean:
 2023-01-01    99.348209
2023-01-08    99.973290
2023-01-15    99.110736
Freq: W-SUN, dtype: float64
7-day rolling mean:
 2023-01-01          NaN
2023-01-02          NaN
2023-01-03          NaN
2023-01-04          NaN
2023-01-05          NaN
2023-01-06          NaN
2023-01-07    99.969472
2023-01-08    99.973290
Freq: D, dtype: float64
['Sunday', 'Monday', 'Tuesday']


**Exercise 15** — From a daily series, compute the month-end (`'ME'`) totals via resampling.

In [44]:
# Your turn:

In [45]:
# Solution
idx = pd.date_range('2023-01-01', periods=120, freq='D')
s = pd.Series(np.ones(120), index=idx)
print(s.resample('ME').sum())   # 'ME' = month-end (pandas 2.2+)

2023-01-31    31.0
2023-02-28    28.0
2023-03-31    31.0
2023-04-30    30.0
Freq: ME, dtype: float64


## 16. apply / map / transform

- `Series.map` — element-wise mapping/lookup.
- `DataFrame.apply` — apply a function along an axis (or to each column).
- `groupby.transform` — return a result aligned to the original rows.

In [46]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
print(df.apply(np.sum, axis=0))           # per column
print(df.apply(np.sum, axis=1))           # per row
print(df['a'].map({1: 'one', 2: 'two', 3: 'three'}))
print(df.apply(lambda row: row['a'] * row['b'], axis=1))   # row-wise custom
# map on a whole frame (pandas 2.1+): element-wise
print(df.map(lambda v: v ** 2))

a     6
b    15
dtype: int64
0    5
1    7
2    9
dtype: int64
0      one
1      two
2    three
Name: a, dtype: str
0     4
1    10
2    18
dtype: int64
   a   b
0  1  16
1  4  25
2  9  36


**Exercise 16** — Map a `grade` column of letters to GPA points (A=4, B=3, C=2, D=1, F=0) and compute the mean GPA.

In [47]:
# Your turn:

In [48]:
# Solution
g = pd.DataFrame({'grade': ['A', 'B', 'A', 'C', 'F', 'B']})
pts = {'A': 4, 'B': 3, 'C': 2, 'D': 1, 'F': 0}
g['gpa'] = g['grade'].map(pts)
print(g)
print("mean GPA:", g['gpa'].mean().round(3))

  grade  gpa
0     A    4
1     B    3
2     A    4
3     C    2
4     F    0
5     B    3
mean GPA: 2.667


## 17. Window Functions (rolling / expanding)

In [49]:
s = pd.Series([1, 2, 3, 4, 5, 6, 7, 8])
print("rolling mean(3):\n", s.rolling(3).mean())
print("rolling sum(3) min_periods=1:\n", s.rolling(3, min_periods=1).sum())
print("expanding max:\n", s.expanding().max())
print("ewm (span=3):\n", s.ewm(span=3).mean().round(3))

rolling mean(3):
 0    NaN
1    NaN
2    2.0
3    3.0
4    4.0
5    5.0
6    6.0
7    7.0
dtype: float64
rolling sum(3) min_periods=1:
 0     1.0
1     3.0
2     6.0
3     9.0
4    12.0
5    15.0
6    18.0
7    21.0
dtype: float64
expanding max:
 0    1.0
1    2.0
2    3.0
3    4.0
4    5.0
5    6.0
6    7.0
7    8.0
dtype: float64
ewm (span=3):
 0    1.000
1    1.667
2    2.429
3    3.267
4    4.161
5    5.095
6    6.055
7    7.031
dtype: float64


**Exercise 17** — Compute a 3-period rolling standard deviation of `[10, 12, 11, 14, 18, 20, 19]`.

In [50]:
# Your turn:

In [51]:
# Solution
s = pd.Series([10, 12, 11, 14, 18, 20, 19])
print(s.rolling(3).std().round(3))

0      NaN
1      NaN
2    1.000
3    1.528
4    3.512
5    3.055
6    1.000
dtype: float64


## 18. MultiIndex (Hierarchical Indexing)

Multiple index levels let you represent higher-dimensional data in a 2-D frame.

In [52]:
idx = pd.MultiIndex.from_product([['2022', '2023'], ['Q1', 'Q2']],
                                 names=['year', 'quarter'])
df = pd.DataFrame({'sales': [100, 120, 140, 160]}, index=idx)
print(df)
print(df.loc['2023'])                      # select outer level
print(df.loc[('2023', 'Q2')])              # select a specific tuple
print(df.groupby(level='year').sum())      # aggregate over a level
print(df.unstack('quarter'))               # pivot a level to columns

              sales
year quarter       
2022 Q1         100
     Q2         120
2023 Q1         140
     Q2         160


         sales
quarter       
Q1         140
Q2         160
sales    160
Name: (2023, Q2), dtype: int64
      sales
year       
2022    220
2023    300
        sales     
quarter    Q1   Q2
year              
2022      100  120
2023      140  160


**Exercise 18** — From the MultiIndexed `df` above, compute the total sales per quarter (across both years).

In [53]:
# Your turn:

In [54]:
# Solution
print(df.groupby(level='quarter').sum())

         sales
quarter       
Q1         240
Q2         280


## 19. Categorical Data

For repeated labels: less memory, faster groupby, and a defined order for sorting.

In [55]:
sizes = pd.Series(['M', 'S', 'L', 'M', 'XL', 'S'])
cat = pd.Categorical(sizes, categories=['S', 'M', 'L', 'XL'], ordered=True)
s = pd.Series(cat)
print(s.sort_values())            # sorts by defined order, not alphabetically
print(s[s > 'M'])                 # ordered comparison works
print(s.cat.codes.tolist())       # integer codes under the hood

1     S
5     S
0     M
3     M
2     L
4    XL
dtype: category
Categories (4, str): ['S' < 'M' < 'L' < 'XL']


2     L
4    XL
dtype: category
Categories (4, str): ['S' < 'M' < 'L' < 'XL']
[1, 0, 2, 1, 3, 0]


**Exercise 19** — Make an ordered categorical of satisfaction `['low','med','high']` and filter rows that are at least `'med'`.

In [56]:
# Your turn:

In [57]:
# Solution
data = pd.Series(['low', 'high', 'med', 'low', 'high'])
c = pd.Categorical(data, categories=['low', 'med', 'high'], ordered=True)
sc = pd.Series(c)
print(sc[sc >= 'med'])

1    high
2     med
4    high
dtype: category
Categories (3, str): ['low' < 'med' < 'high']


## 20. File I/O

CSV is the lingua franca; parquet is fast+typed for larger data.

In [58]:
import io
csv_text = "name,age,city\nAlice,30,NYC\nBob,25,LA\nCara,35,SF\n"
df = pd.read_csv(io.StringIO(csv_text))
print(df)

# Round-trip to CSV in memory
buf = io.StringIO()
df.to_csv(buf, index=False)
print("CSV out:\n", buf.getvalue())

# JSON round-trip
js = df.to_json(orient='records')
print("JSON:", js)
print(pd.read_json(io.StringIO(js)))

# read_csv superpowers
tricky = "id;val\n1;3,14\n2;2,71\n"
print(pd.read_csv(io.StringIO(tricky), sep=';', decimal=','))

    name  age city
0  Alice   30  NYC
1    Bob   25   LA
2   Cara   35   SF
CSV out:
 name,age,city
Alice,30,NYC
Bob,25,LA
Cara,35,SF

JSON: [{"name":"Alice","age":30,"city":"NYC"},{"name":"Bob","age":25,"city":"LA"},{"name":"Cara","age":35,"city":"SF"}]
    name  age city
0  Alice   30  NYC
1    Bob   25   LA
2   Cara   35   SF


   id   val
0   1  3.14
1   2  2.71


**Exercise 20** — Read a CSV where missing values are the string `'NA'` and `'-'`, treating both as NaN. (Hint: `na_values`.)

In [59]:
# Your turn:

In [60]:
# Solution
import io
txt = "a,b\n1,NA\n-,4\n3,5\n"
df = pd.read_csv(io.StringIO(txt), na_values=['NA', '-'])
print(df)
print(df.isna().sum())

     a    b
0  1.0  NaN
1  NaN  4.0
2  3.0  5.0
a    1
b    1
dtype: int64


## 21. Performance

Vectorize; avoid `iterrows`. Use built-in methods, `query`/`eval` for big expressions,
and categoricals for repeated strings.

In [61]:
import time
n = 200_000
df = pd.DataFrame({'a': np.arange(n), 'b': np.arange(n)})

t = time.perf_counter()
res_loop = [a + b for a, b in zip(df['a'], df['b'])]   # python-level
loop_t = time.perf_counter() - t

t = time.perf_counter()
res_vec = df['a'] + df['b']                             # vectorized
vec_t = time.perf_counter() - t

print(f"loop  : {loop_t*1000:6.1f} ms")
print(f"vector: {vec_t*1000:6.1f} ms  -> {loop_t/vec_t:.0f}x faster")
print("results match:", np.array_equal(np.array(res_loop), res_vec.values))

loop  :   14.4 ms
vector:    0.3 ms  -> 43x faster
results match: True


**Exercise 21** — Replace a slow row-wise `apply` that computes `a*b + c` with a single vectorized expression and confirm equality.

In [62]:
# Your turn:

In [63]:
# Solution
rng = np.random.default_rng(6)
df = pd.DataFrame(rng.integers(0, 10, size=(1000, 3)), columns=['a', 'b', 'c'])
slow = df.apply(lambda r: r['a'] * r['b'] + r['c'], axis=1)
fast = df['a'] * df['b'] + df['c']
print("equal:", slow.equals(fast))

equal:

 True


## 22. 🏆 Capstone

Combine everything. Try each before opening the solution.

**Capstone A — End-to-end analysis.** Given a synthetic transactions table (date, region, product, amount): (1) parse dates, (2) monthly total revenue per region, (3) the single best (region, product) pair by total revenue.

In [64]:
# Your turn:

In [65]:
# Solution
rng = np.random.default_rng(20)
n = 500
tx = pd.DataFrame({
    'date': pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 365, n), 'D'),
    'region': rng.choice(['N', 'S', 'E', 'W'], n),
    'product': rng.choice(['x', 'y', 'z'], n),
    'amount': rng.integers(10, 500, n),
})
tx['month'] = tx['date'].dt.to_period('M')
monthly = tx.groupby(['month', 'region'])['amount'].sum().unstack()
print(monthly.head())
best = tx.groupby(['region', 'product'])['amount'].sum().idxmax()
print("best (region, product):", best)

region      E     N     S     W
month                          
2023-01  1055  2779  2285  3293
2023-02  2824  2297  1521  2028
2023-03  4066  2853  4160  2697
2023-04  2538  3660  2575  2353
2023-05  2913  3383  3517  2354


best (region, product): ('E', 'x')


**Capstone B — Clean + feature engineer.** Given a messy frame with missing ages and mixed-case cities: (1) fill missing age with the median, (2) normalize city to title case, (3) bucket age into `['<30','30-50','>50']`, (4) count rows per bucket.

In [66]:
# Your turn:

In [67]:
# Solution
df = pd.DataFrame({'age': [25, np.nan, 55, 40, np.nan, 33],
                   'city': ['nyc', 'LA', 'sf', 'NYC', 'la', 'SF']})
df['age'] = df['age'].fillna(df['age'].median())
df['city'] = df['city'].str.upper()
df['bucket'] = pd.cut(df['age'], bins=[0, 30, 50, 200],
                      labels=['<30', '30-50', '>50'], right=False)
print(df)
print(df['bucket'].value_counts())

    age city bucket
0  25.0  NYC    <30
1  36.5   LA  30-50
2  55.0   SF    >50
3  40.0  NYC  30-50
4  36.5   LA  30-50
5  33.0   SF  30-50
bucket
30-50    4
<30      1
>50      1
Name: count, dtype: int64


**Capstone C — Join + rank.** Two tables (employees with dept_id; departments with name). Join them, then within each department rank employees by salary and keep the top earner per department.

In [68]:
# Your turn:

In [69]:
# Solution
emp = pd.DataFrame({'emp': list('abcdef'),
                    'dept_id': [1, 1, 2, 2, 2, 3],
                    'salary': [90, 85, 120, 110, 95, 70]})
dept = pd.DataFrame({'dept_id': [1, 2, 3], 'dept_name': ['eng', 'sales', 'hr']})
j = emp.merge(dept, on='dept_id')
j['rank'] = j.groupby('dept_name')['salary'].rank(ascending=False, method='min')
print(j.sort_values(['dept_name', 'rank']))
print("\nTop earner per dept:")
print(j[j['rank'] == 1][['dept_name', 'emp', 'salary']])

  emp  dept_id  salary dept_name  rank
0   a        1      90       eng   1.0
1   b        1      85       eng   2.0
5   f        3      70        hr   1.0
2   c        2     120     sales   1.0
3   d        2     110     sales   2.0
4   e        2      95     sales   3.0

Top earner per dept:


  dept_name emp  salary
0       eng   a      90
2     sales   c     120
5        hr   f      70


---
## 🎓 Next Steps
- Redo the exercises from scratch in a week (delete the solutions first).
- Learn method chaining (`.pipe`, `.assign`, `.loc[...]`) for readable pipelines.
- Pair with **matplotlib**/**seaborn** (next notebooks) to visualize results.